# LifeMate — lm-A-030: fine-tune ASR Whisper trên audio train

**Import notebook này, bật GPU T4 x2 và Internet On.** Chỉ cần dataset cuộc thi (đã gắn sẵn trong metadata). Ước tính 2–4 giờ.

Vì sao: run lm-B-022 (oracle) cho thấy checkpoint 021 đạt **63.13** với transcript gold nhưng chỉ **36.63** với gipformer, tức ASR làm mất ~26.5 điểm dev.

Làm gì: fine-tune Whisper trên audio `train_fit` (2955 audio, gồm cả lượt history), mỗi GPU một biến thể:
- GPU0 `small` = `vinai/PhoWhisper-small` (AdamW)
- GPU1 `medium` = `vinai/PhoWhisper-medium` (AdamW 8-bit, gradient checkpointing)

Chọn epoch theo WER trên dev (1627 audio của đúng 417 mẫu dev mà các run agent dùng), báo WER `final_holdout` một lần, rồi chép transcript dev + holdout + toàn bộ public_test bằng epoch tốt nhất. Target là `audio_text` gốc (có hoa, dấu câu, chữ số), nên model học luôn cách viết số.

Mốc so sánh: gipformer, WER lượt cuối dev **19.58%** (seen 19.81%, unseen 19.44%; tính local từ traces 022).

Output `/kaggle/working/asr_ft/<variant>/`: `metrics.json`, `asr_train_<variant>.shard0.jsonl` (dev + holdout), `asr_public_test_<variant>.shard0.jsonl`, `model/` (fp16), `worker.log`.

In [ ]:
CONFIG = {
    "experiment": "lm-A-030-asr-ft-whisper",
    "seed": 2026,
    "out_dir": "/kaggle/working/asr_ft",
    "loader_workers": 2,            # mỗi worker; máy Kaggle có 4 CPU
    "session_budget_s": 11.5 * 3600,
    "reserve_s": 1800,              # chừa cho holdout + public + lưu model sau epoch cuối
    "max_new_tokens": 160,
    "eval_zero_shot": True,         # epoch 0 = model gốc, để so trước/sau
    "save_model": True,
    "variants": {
        "small": {"gpu": 0, "model": "vinai/PhoWhisper-small", "revision": None, "lr": 1.25e-5, "bs": 16, "accum": 1,
                  "epochs": 5, "eval_bs": 32, "optim": "adamw", "grad_ckpt": False},
        "medium": {"gpu": 1, "model": "vinai/PhoWhisper-medium", "revision": "55a7e3eb6c906de891f8f06a107754427dd3be79",
                   "lr": 6.25e-6, "bs": 8, "accum": 2, "epochs": 5, "eval_bs": 16, "optim": "adamw8bit", "grad_ckpt": True},
    },
}

In [ ]:
import glob, json, os, socket, subprocess, sys, time
from pathlib import Path
SESSION_STARTED = time.time()
WORK = Path("/kaggle/working"); OUT = Path(CONFIG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-2000:]); return r.returncode

# Model tải từ Hugging Face: thiếu mạng thì dừng ngay (lần 1 mất 41 phút retry rồi mới chết)
try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
except OSError as e:
    raise RuntimeError("Không có Internet: Session options → Internet On, rồi Save & Run All lại") from e
from huggingface_hub import snapshot_download
for v in CONFIG["variants"].values():
    print("tải", v["model"], snapshot_download(v["model"], revision=v["revision"]))

# Gói nhẹ: cài thẳng vào môi trường gốc; --no-deps để không đụng torch. Lỗi thì worker tự dùng Adafactor.
sh(f"{sys.executable} -m pip install -q --no-deps bitsandbytes")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__, "gpus", torch.cuda.device_count())

def find_split(split):
    hits = sorted(glob.glob(f"/kaggle/input/**/{split}/{split}.jsonl", recursive=True))
    if not hits: raise FileNotFoundError(f"không thấy {split}.jsonl: Add Input dataset thanhkhil/zaloai-lifemate-dataset")
    return Path(hits[0]).parent
TRAIN_DIR, PUBLIC_DIR = find_split("train"), find_split("public_test")
print(TRAIN_DIR, PUBLIC_DIR)

In [ ]:
%%writefile /kaggle/working/asrft.py
"""Whisper fine-tune worker, one variant per GPU: train on train_fit audio (history + final turns), pick the epoch
by dev WER, report final_holdout WER once, then transcribe dev / holdout / public_test with the chosen epoch.

    python asrft.py --config cfg.json --variant small
Split = the same seed-2026 manifest as lmlib.data.make_manifest (copied verbatim below).
"""
from __future__ import annotations

import argparse
import hashlib
import json
import math
import random
import re
import sys
import time
import unicodedata
from collections import Counter
from pathlib import Path


# ------------------------------------------------------------------ split (verbatim from lmlib/data.py)
def audio_paths(row: dict) -> list[str]:
    out = [t["audio"] for t in row.get("history", []) if t.get("role") == "user" and t.get("audio")]
    return out + [row["audio"]]


def _template_key(row: dict) -> str:
    text = re.sub(r"\d+", "#", (row.get("audio_text") or "").lower())
    names = ",".join(sorted({c["name"] for c in (row.get("label") or {}).get("tool_calls", [])}))
    return hashlib.md5(f"{names}|{text[:40]}".encode()).hexdigest()


def label_apps(row: dict) -> set[str]:
    calls = list((row.get("label") or {}).get("tool_calls", []))
    for t in row.get("history", []):
        calls += t.get("tool_calls", []) or []
    return {c["name"].split(".", 1)[0] for c in calls if not c["name"].startswith("sys.")}


def make_manifest(rows: list[dict], seed: int = 2026, holdout_apps: int = 4, dev_frac: float = 0.12, final_frac: float = 0.12) -> dict:
    rng = random.Random(seed)
    use = Counter(a for r in rows for a in label_apps(r))
    cands = sorted(a for a, n in use.items() if 20 <= n <= 120)
    held = sorted(rng.sample(cands, min(holdout_apps, len(cands))))
    unseen = [r["id"] for r in rows if label_apps(r) & set(held)]
    rest = [r for r in rows if r["id"] not in set(unseen)]
    groups: dict[str, list[str]] = {}
    for r in rest:
        groups.setdefault(_template_key(r), []).append(r["id"])
    keys = sorted(groups)
    rng.shuffle(keys)
    n = len(rest)
    ids = {"train_fit": [], "dev_seen": [], "final_holdout": [], "dev_unseen": unseen}
    for k in keys:
        if len(ids["final_holdout"]) < final_frac * n:
            ids["final_holdout"] += groups[k]
        elif len(ids["dev_seen"]) < dev_frac * n:
            ids["dev_seen"] += groups[k]
        else:
            ids["train_fit"] += groups[k]
    return {"seed": seed, "holdout_apps": held, "ids": ids, "counts": {k: len(v) for k, v in ids.items()}}


def build_items(train_rows: list[dict], public_rows: list[dict]) -> dict:
    """{part: [{audio, text, episode, turn, part}]}; every user audio of an episode goes to the episode's part."""
    m = make_manifest(sorted(train_rows, key=lambda r: r["id"]))
    part_of = {i: p for p, ids in m["ids"].items() for i in ids}
    parts = {"train_fit": [], "dev_seen": [], "dev_unseen": [], "final_holdout": [], "public": []}
    seen = {}
    for r in train_rows:
        turns = [t for t in r.get("history", []) if t.get("role") == "user" and t.get("audio")] + [r]
        for k, t in enumerate(turns):
            turn = "final" if k == len(turns) - 1 else "history"
            if t["audio"] in seen:
                raise ValueError(f"audio shared by two turns: {t['audio']}")
            seen[t["audio"]] = 1
            parts[part_of[r["id"]]].append({"audio": t["audio"], "text": t["audio_text"], "episode": r["id"], "turn": turn,
                                            "part": part_of[r["id"]]})
    for r in public_rows:
        for p in audio_paths(r):
            parts["public"].append({"audio": p, "text": None, "episode": r["id"], "turn": "final" if p == r["audio"] else "history",
                                    "part": "public"})
    return {"manifest_counts": m["counts"], "holdout_apps": m["holdout_apps"], "parts": parts}


# ------------------------------------------------------------------ metrics (as lmlib/asr.py)
def norm_text(s: str) -> str:
    t = unicodedata.normalize("NFC", s or "").lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(t.split())


def edit_distance(a: list, b: list) -> int:
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


def score(items: list[dict], hyp: dict[str, str]) -> dict:
    """Corpus WER / CER and normalised exact-sentence rate, overall and per (part, turn) slice."""
    def agg(sel):
        we = wn = ce = cn = ex = 0
        for it in sel:
            g, h = norm_text(it["text"]), norm_text(hyp.get(it["audio"], ""))
            we += edit_distance(g.split(), h.split()); wn += len(g.split())
            ce += edit_distance(list(g.replace(" ", "")), list(h.replace(" ", ""))); cn += len(g.replace(" ", ""))
            ex += g == h
        return {"n": len(sel), "wer": we / max(wn, 1), "cer": ce / max(cn, 1), "exact": ex / max(len(sel), 1)}
    out = {"all": agg(items), "final": agg([i for i in items if i["turn"] == "final"])}
    for part in sorted({i["part"] for i in items}):
        out[f"{part}/final"] = agg([i for i in items if i["part"] == part and i["turn"] == "final"])
    return out


# ------------------------------------------------------------------ audio / batches
def read_wave(path: Path):
    import numpy as np
    import soundfile as sf

    w, sr = sf.read(str(path), dtype="float32")
    if w.ndim > 1:
        w = w.mean(axis=1)
    if sr != 16000:
        import librosa

        w = librosa.resample(w, orig_sr=sr, target_sr=16000)
    return np.ascontiguousarray(w)


class AudioSet:
    """Log-mel features (+ label ids when `tok` is given) computed in DataLoader workers."""

    def __init__(self, items, root: Path, fe, tok=None):
        self.items, self.root, self.fe, self.tok = items, root, fe, tok

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        feat = self.fe(read_wave(self.root / it["audio"]), sampling_rate=16000, return_tensors="np").input_features[0]
        if self.tok is None:
            return feat, i
        return feat, self.tok(it["text"]).input_ids


def collate_train(batch, start_id):
    import numpy as np
    import torch

    feats = torch.from_numpy(np.stack([b[0] for b in batch]))
    labels = torch.full((len(batch), max(len(b[1]) for b in batch)), -100, dtype=torch.long)
    for i, (_, lab) in enumerate(batch):
        labels[i, : len(lab)] = torch.tensor(lab)
    if (labels[:, 0] == start_id).all():  # the model prepends decoder_start_token_id itself
        labels = labels[:, 1:]
    return feats, labels


def collate_eval(batch):
    import numpy as np
    import torch

    return torch.from_numpy(np.stack([b[0] for b in batch])), [b[1] for b in batch]


def transcribe(model, fe, tok, items, root, bs, max_new_tokens, workers) -> tuple[dict[str, str], float]:
    import torch

    model.eval()
    dl = torch.utils.data.DataLoader(AudioSet(items, root, fe), batch_size=bs, shuffle=False, num_workers=workers,
                                     collate_fn=collate_eval)
    out, t0 = {}, time.time()
    with torch.no_grad(), autocast():
        for feats, idx in dl:
            ids = model.generate(input_features=feats.to(DEVICE), language="vi", task="transcribe", max_new_tokens=max_new_tokens,
                                 num_beams=1, do_sample=False, use_cache=True)
            for i, text in zip(idx, tok.batch_decode(ids, skip_special_tokens=True)):
                out[items[i]["audio"]] = text.strip()
    return out, (time.time() - t0) / max(len(items), 1)


DEVICE = "cuda"  # main() switches to cpu for the local smoke test


def autocast():
    import torch

    return torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda")


def make_optimizer(model, v):
    params = [p for p in model.parameters() if p.requires_grad]
    if v.get("optim") == "adamw8bit":
        try:
            import bitsandbytes as bnb

            return bnb.optim.AdamW8bit(params, lr=v["lr"], weight_decay=0.01), "adamw8bit"
        except Exception as e:  # no bitsandbytes / no CUDA kernel: factored optimizer keeps memory low
            print(f"[asrft] bitsandbytes unavailable ({type(e).__name__}: {e}); using Adafactor", flush=True)
            from transformers.optimization import Adafactor

            return Adafactor(params, lr=v["lr"], scale_parameter=False, relative_step=False, warmup_init=False,
                             weight_decay=0.01), "adafactor"
    import torch

    return torch.optim.AdamW(params, lr=v["lr"], weight_decay=0.01), "adamw"


def write_cache(path: Path, tag: str, items, hyp):
    """Same layout as lmlib.asr legacy caches (header row, then one record per audio), readable by load_cache."""
    with open(path, "w", encoding="utf-8") as f:
        f.write(json.dumps({"_header": {"run_hash": tag, "created": time.time()}}) + "\n")
        for it in items:
            f.write(json.dumps({"id": it["audio"], "status": "completed", "text": hyp[it["audio"]]}, ensure_ascii=False) + "\n")


# ------------------------------------------------------------------ worker
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--variant", required=True)
    a = ap.parse_args(argv)
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    v = cfg["variants"][a.variant]

    import numpy as np
    import torch
    from transformers import WhisperForConditionalGeneration, WhisperProcessor, get_linear_schedule_with_warmup

    global DEVICE
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    from transformers.utils import logging as hf_logging

    hf_logging.set_verbosity_error()  # per-batch generate() deprecation notes would flood the worker log

    random.seed(cfg["seed"]); np.random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])
    out = Path(cfg["out_dir"]) / a.variant
    out.mkdir(parents=True, exist_ok=True)
    split = json.loads(Path(cfg["split_json"]).read_text(encoding="utf-8"))["parts"]
    train_root, public_root = Path(cfg["train_dir"]), Path(cfg["public_dir"])
    dev = split["dev_seen"] + split["dev_unseen"]
    workers = int(cfg.get("loader_workers", 2))
    deadline = cfg["session_started"] + cfg["session_budget_s"]
    metrics = {"variant": a.variant, "config": v, "epochs": []}

    def save_metrics():
        (out / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=1), encoding="utf-8")

    t = time.time()
    proc = WhisperProcessor.from_pretrained(v["model"], revision=v.get("revision"))
    tok, fe = proc.tokenizer, proc.feature_extractor
    tok.set_prefix_tokens(language="vietnamese", task="transcribe", predict_timestamps=False)
    model = WhisperForConditionalGeneration.from_pretrained(v["model"], revision=v.get("revision"), torch_dtype=torch.float32).to(DEVICE)
    model.generation_config.forced_decoder_ids = None
    model.config.forced_decoder_ids = None
    model.config.apply_spec_augment = True
    model.config.mask_time_prob, model.config.mask_time_length = v.get("mask_time_prob", 0.05), 10
    model.config.mask_feature_prob, model.config.mask_feature_length = v.get("mask_feature_prob", 0.05), 10
    n_params = sum(p.numel() for p in model.parameters())
    if n_params + 4022468096 >= 6_000_000_000:
        raise ValueError("LifeMate total parameter limit exceeded")
    metrics.update(model_load_s=time.time() - t, parameter_count=n_params,
                   revision=getattr(model.config, "_commit_hash", None), n_train=len(split["train_fit"]), n_dev=len(dev))
    print(f"[asrft {a.variant}] loaded {n_params / 1e6:.0f}M params in {metrics['model_load_s']:.0f}s; "
          f"train {len(split['train_fit'])} audio, dev {len(dev)}", flush=True)

    def eval_dev(epoch, train_s=None, loss=None):
        hyp, s_per = transcribe(model, fe, tok, dev, train_root, v["eval_bs"], cfg["max_new_tokens"], workers)
        rec = {"epoch": epoch, "dev": score(dev, hyp), "s_per_audio": s_per, "train_s": train_s, "train_loss": loss,
               "peak_vram_mb": torch.cuda.max_memory_allocated() / 2**20 if DEVICE == "cuda" else None}
        metrics["epochs"].append(rec); save_metrics()
        d = rec["dev"]
        print(f"[asrft {a.variant}] epoch {epoch}: dev WER all {d['all']['wer']:.4f} final {d['final']['wer']:.4f} "
              f"(seen {d['dev_seen/final']['wer']:.4f} unseen {d['dev_unseen/final']['wer']:.4f}) {s_per:.3f}s/audio", flush=True)
        return d["all"]["wer"], hyp

    best_wer, best_hyp = eval_dev(0) if cfg.get("eval_zero_shot", True) else (float("inf"), None)
    best_epoch, best_state = 0, None

    if v.get("grad_ckpt"):
        model.gradient_checkpointing_enable()
    model.config.use_cache = False
    opt, opt_name = make_optimizer(model, v)
    metrics["optimizer"] = opt_name
    g = torch.Generator(); g.manual_seed(cfg["seed"])
    dl = torch.utils.data.DataLoader(AudioSet(split["train_fit"], train_root, fe, tok), batch_size=v["bs"], shuffle=True,
                                     generator=g, num_workers=workers, drop_last=True,
                                     collate_fn=lambda b: collate_train(b, model.config.decoder_start_token_id))
    accum = int(v.get("accum", 1))
    total = v["epochs"] * (len(dl) // accum)
    sched = get_linear_schedule_with_warmup(opt, min(v.get("warmup", 50), total // 10), total)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
    epoch_s = None
    for ep in range(1, v["epochs"] + 1):
        if epoch_s and time.time() + 1.3 * epoch_s + cfg["reserve_s"] > deadline:
            print(f"[asrft {a.variant}] stop before epoch {ep}: session budget", flush=True)
            metrics["stopped_by_budget"] = ep; break
        model.train(); te = time.time(); losses = []
        opt.zero_grad(set_to_none=True)
        for step, (feats, labels) in enumerate(dl):
            with autocast():
                loss = model(input_features=feats.to(DEVICE), labels=labels.to(DEVICE)).loss / accum
            scaler.scale(loss).backward()
            losses.append(loss.item() * accum)
            if (step + 1) % accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True)
            if step % 50 == 0:
                print(f"[asrft {a.variant}] ep {ep} step {step}/{len(dl)} loss {np.mean(losses[-50:]):.4f}", flush=True)
        model.config.use_cache = True
        wer, hyp = eval_dev(ep, time.time() - te, float(np.mean(losses)))
        model.config.use_cache = False
        epoch_s = time.time() - te
        if wer < best_wer:
            best_wer, best_epoch, best_hyp = wer, ep, hyp
            best_state = {k: t.detach().to("cpu", torch.float16, copy=True) for k, t in model.state_dict().items()}

    metrics.update(best_epoch=best_epoch, best_dev_wer=best_wer)
    if best_state is not None:
        model.load_state_dict({k: t.float() for k, t in best_state.items()})
    elif metrics["epochs"][-1]["epoch"] > 0:  # zero-shot stayed best: back to the pretrained weights
        model = WhisperForConditionalGeneration.from_pretrained(v["model"], revision=v.get("revision"), torch_dtype=torch.float32).to(DEVICE)
        model.generation_config.forced_decoder_ids = None
    model.config.use_cache = True
    hold, _ = transcribe(model, fe, tok, split["final_holdout"], train_root, v["eval_bs"], cfg["max_new_tokens"], workers)
    metrics["holdout"] = score(split["final_holdout"], hold)
    pub, s_pub = transcribe(model, fe, tok, split["public"], public_root, v["eval_bs"], cfg["max_new_tokens"], workers)
    metrics["public_s_per_audio"] = s_pub
    tag = f"asrft-{a.variant}-ep{best_epoch}"
    write_cache(out / f"asr_train_{a.variant}.shard0.jsonl", tag, dev + split["final_holdout"], {**best_hyp, **hold})
    write_cache(out / f"asr_public_test_{a.variant}.shard0.jsonl", tag, split["public"], pub)
    if best_state is not None and cfg.get("save_model", True):
        model.half().save_pretrained(out / "model", safe_serialization=True)
        proc.save_pretrained(out / "model")
    metrics["total_s"] = time.time() - cfg["session_started"]
    save_metrics()
    h = metrics["holdout"]
    print(f"[asrft {a.variant}] best epoch {best_epoch}: dev WER {best_wer:.4f}; holdout WER all {h['all']['wer']:.4f} "
          f"final {h['final']['wer']:.4f}; public {len(pub)} audio at {s_pub:.3f}s/audio", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
sys.path.insert(0, str(WORK))
import asrft
load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]
split = asrft.build_items(load(TRAIN_DIR / "train.jsonl"), load(PUBLIC_DIR / "public_test.jsonl"))
# Cùng split seed 2026 với mọi run agent (lmlib.data.make_manifest)
assert split["manifest_counts"] == {"train_fit": 974, "dev_seen": 154, "final_holdout": 154, "dev_unseen": 263}, split["manifest_counts"]
print(split["holdout_apps"], {k: len(v) for k, v in split["parts"].items()})
(OUT / "split.json").write_text(json.dumps(split, ensure_ascii=False), encoding="utf-8")
cfg = dict(CONFIG, split_json=str(OUT / "split.json"), train_dir=str(TRAIN_DIR), public_dir=str(PUBLIC_DIR),
           session_started=SESSION_STARTED)
(OUT / "config.json").write_text(json.dumps(cfg, indent=1), encoding="utf-8")

## Train + eval (một worker / GPU)

In [ ]:
# Mỗi GPU một worker, chạy song song; log ở asr_ft/<variant>/worker.log
procs = {}
for name, v in CONFIG["variants"].items():
    (OUT / name).mkdir(exist_ok=True)
    log = open(OUT / name / "worker.log", "a", encoding="utf-8")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(v["gpu"]), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    procs[name] = subprocess.Popen([sys.executable, str(WORK / "asrft.py"), "--config", str(OUT / "config.json"), "--variant", name],
                                   env=env, stdout=log, stderr=subprocess.STDOUT, cwd=str(WORK))
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    for name in procs:
        lines = [l for l in open(OUT / name / "worker.log", encoding="utf-8", errors="replace") if l.startswith("[asrft")]
        print(f"{time.time() - SESSION_STARTED:7.0f}s {name}: {lines[-1].strip() if lines else '(đang load)'}", flush=True)
for name, p in procs.items():
    print(name, "exit", p.returncode)
    if p.returncode:
        print("".join(open(OUT / name / "worker.log", encoding="utf-8", errors="replace").readlines()[-40:]))

## Kết quả

In [ ]:
# Bảng WER dev theo epoch (gipformer lượt cuối dev = 0.1958) + holdout của epoch được chọn
GIPFORMER_DEV_FINAL = 0.1958
for name in CONFIG["variants"]:
    p = OUT / name / "metrics.json"
    if not p.exists(): print(name, "không có metrics"); continue
    m = json.loads(p.read_text())
    print(f"== {name}: {m.get('parameter_count', 0) / 1e6:.0f}M params, optimizer {m.get('optimizer')}, best epoch {m.get('best_epoch')}")
    print(" ep | WER all | WER final | seen final | unseen final | exact final | s/audio | train s | peak MB")
    for e in m["epochs"]:
        d = e["dev"]
        print(f" {e['epoch']:2d} | {d['all']['wer']:.4f}  | {d['final']['wer']:.4f}    | {d['dev_seen/final']['wer']:.4f}     | "
              f"{d['dev_unseen/final']['wer']:.4f}       | {d['final']['exact']:.3f}       | {e['s_per_audio']:.3f}   | "
              f"{(e['train_s'] or 0):7.0f} | {(e['peak_vram_mb'] or 0):.0f}")
    if "holdout" in m:
        h = m["holdout"]
        print(f" holdout: WER all {h['all']['wer']:.4f}, final {h['final']['wer']:.4f}, exact final {h['final']['exact']:.3f}; "
              f"public {m.get('public_s_per_audio', 0):.3f}s/audio; tổng {m.get('total_s', 0) / 60:.0f} phút")
    best = [e for e in m["epochs"] if e["epoch"] == m.get("best_epoch")]
    if best:
        print(f" lượt cuối dev: {best[0]['dev']['final']['wer']:.4f} vs gipformer {GIPFORMER_DEV_FINAL}")

In [ ]:
# Câu dev lượt cuối sai nhiều nhất của từng biến thể (epoch được chọn)
for name in CONFIG["variants"]:
    p = OUT / name / f"asr_train_{name}.shard0.jsonl"
    if not p.exists(): continue
    hyp = {r["id"]: r["text"] for r in load(p)[1:]}
    rows = []
    for it in split["parts"]["dev_seen"] + split["parts"]["dev_unseen"]:
        if it["turn"] != "final": continue
        g = asrft.norm_text(it["text"]).split(); h = asrft.norm_text(hyp.get(it["audio"], "")).split()
        rows.append((asrft.edit_distance(g, h) / max(len(g), 1), it["text"], hyp.get(it["audio"], "")))
    print(f"== {name}")
    for w, g, h in sorted(rows, reverse=True)[:12]:
        print(f"{w:.2f}\n  G: {g}\n  A: {h}")